## Shared OpenAI Helpers
Shared API calls, validation, retry logic, and JSON saving.

In [338]:
import json
import os
import random
import re
from pathlib import Path

import google.generativeai as genai
import requests
from dotenv import load_dotenv


def _load_openai_api_key():
    for candidate in [Path.cwd() / "PROMPTFOO" / ".env", Path.cwd() / ".env"]:
        if candidate.exists():
            load_dotenv(candidate, override=True)
            break
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY was not found in the workspace .env file")
    return api_key


OPENAI_API_KEY = _load_openai_api_key()
OPENAI_CHAT_ENDPOINT = "https://api.openai.com/v1/chat/completions"
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GOOGLE_MODEL = "gemini-2.5-flash"

if not GOOGLE_API_KEY:
    raise RuntimeError("GOOGLE_API_KEY was not found in the workspace .env file")

genai.configure(api_key=GOOGLE_API_KEY)


def extract_json_payload(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise


def call_openai_json(prompt, model="gpt-4o-mini", temperature=0.8):
    response = requests.post(
        OPENAI_CHAT_ENDPOINT,
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": "Return raw JSON only. No markdown, no commentary."},
                {"role": "user", "content": prompt},
            ],
            "temperature": temperature,
            "response_format": {"type": "json_object"},
        },
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    return extract_json_payload(content)


def call_google_json(prompt, model=GOOGLE_MODEL, temperature=0):
    model_client = genai.GenerativeModel(model)
    response = model_client.generate_content(
        prompt,
        generation_config={
            "temperature": temperature,
            "response_mime_type": "application/json",
        },
    )
    text = getattr(response, "text", "")
    if isinstance(text, dict):
        return text

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise


def save_json_output(folder, filename, payload):
    os.makedirs(folder, exist_ok=True)
    output_path = Path(folder) / filename
    with output_path.open("w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, ensure_ascii=False, indent=2)
    return str(output_path)


def _infer_activity_kind(filename):
    lower_name = filename.lower()
    if "pattern" in lower_name:
        return "pattern"
    if "sequencing" in lower_name:
        return "sequencing"
    if "logic" in lower_name:
        return "logic"
    return "unknown"


def _infer_level(filename):
    lower_name = filename.lower()
    for level_name in ("beginner", "intermediate", "hard"):
        if level_name in lower_name:
            return level_name
    return "unknown"


def _build_generation_prompt(activity_kind, level_name, prompt_text, attempt):
    base_context = [
        f"Activity: {activity_kind}.",
        f"Difficulty: {level_name}.",
        f"This is a {level_name} preschool {activity_kind} task.",
    ]

    if activity_kind == "logic":
        base_context.append("Return JSON with keys grid_colors, diff_index, and diff_color.")
        if level_name == "beginner":
            base_context.append("Use varied child-friendly palettes (not the same standard RGBY/neon look every time).")
            base_context.append("When choosing diff_color, compare it only with grid_colors[diff_index] (the replaced slot color).")
    elif activity_kind == "sequencing":
        base_context.append("Return JSON with keys jumbled_sequence and correct_index_order.")
        base_context.append("For every item, derive its rank from size_rating in the SAME shown order: 1=smallest, N=largest.")
        base_context.append("This is dynamic for any values: sort the size_rating values ascending, map each item to its rank, and output those ranks in shown order.")
        base_context.append("Example only: if size_rating is [3, 5, 1], correct_index_order must be [2, 3, 1].")
        base_context.append("Do not return position-order format like [3, 1, 2].")
    elif activity_kind == "pattern":
        base_context.append("Return JSON with key sequence only.")
        base_context.append("sequence must be the COMPLETE pattern with NO question mark and NO blank.")
        base_context.append("Do not output correct_answer.")
        base_context.append("The app will compute the blank and correct answer from your last sequence item.")
        base_context.append(f"Allowed families for this level: {PATTERN_ALLOWED_FAMILIES.get(level_name, PATTERN_ALLOWED_FAMILIES['beginner'])}. Pick one and stay inside that set.")
    else:
        base_context.append("Return JSON suitable for the requested activity.")

    if attempt > 0:
        base_context.append("Create a different valid variation than prior attempts.")

    context_text = " ".join(base_context)
    return f"{prompt_text}\n\n{context_text}"


# -------------------- DETERMINISTIC VALIDATORS --------------------
# These are strict, code-based checks (not LLM judgment) used before Gemini.

def _validate_logic_payload(payload, level_name):
    colors = payload.get("grid_colors")
    diff_index = payload.get("diff_index")
    diff_color = payload.get("diff_color")

    if not isinstance(colors, list) or not colors:
        return False, "grid_colors must be a non-empty list."
    if any(not isinstance(color, str) or not color.strip() for color in colors):
        return False, "grid_colors must contain non-empty color strings."
    if not isinstance(diff_index, int) or diff_index < 0 or diff_index >= len(colors):
        return False, "diff_index is missing or out of range."
    if not isinstance(diff_color, str) or not diff_color.strip():
        return False, "diff_color must be a non-empty string."

    replaced_color = colors[diff_index]
    if diff_color.strip().lower() == str(replaced_color).strip().lower():
        return False, "diff_color must differ from the grid color at diff_index."

    return True, ""


def _validate_pattern_payload(payload, level_name):
    sequence = payload.get("sequence")
    correct_answer = payload.get("correct_answer")
    if not isinstance(sequence, list) or not sequence:
        return False, "sequence must be a non-empty list."

    expected_lengths = {"beginner": 5, "intermediate": 6, "hard": 8}
    expected_length = expected_lengths.get(level_name)
    if expected_length is None:
        return False, "the level could not be identified for pattern validation."
    if len(sequence) != expected_length:
        return False, f"sequence length is not valid for {level_name}."

    if any(not isinstance(item, str) or not item.strip() for item in sequence):
        return False, "sequence items must be non-empty strings."
    if sequence.count("?") != 1:
        return False, "sequence must contain exactly one question mark."

    if not isinstance(correct_answer, str) or not correct_answer.strip():
        return False, "correct_answer must be a non-empty string."
    if correct_answer.strip() == "?":
        return False, "correct_answer cannot be '?'."

    unique_non_blank = []
    for item in sequence:
        if item != "?" and item not in unique_non_blank:
            unique_non_blank.append(item)
    required_unique_by_level = {"beginner": 2, "intermediate": 2, "hard": 3}
    required_unique = required_unique_by_level.get(level_name, 2)
    if len(unique_non_blank) < required_unique:
        return False, f"{level_name} pattern must include at least {required_unique} unique non-blank items."

    full_sequence = [correct_answer if item == "?" else item for item in sequence]
    symbol_map = {}
    next_codepoint = ord("A")
    signature_chars = []
    for token in full_sequence:
        if token not in symbol_map:
            symbol_map[token] = chr(next_codepoint)
            next_codepoint += 1
        signature_chars.append(symbol_map[token])
    signature = "".join(signature_chars)

    allowed_signatures_by_level = {
        "beginner": {"ABABA", "AABBA"},
        "intermediate": {"AABAAB", "ABBABB", "ABCABC", "AABBAA"},
        "hard": {"ABACABAC", "ABCABCAB", "AABCAABC", "ABBCABBC"},
    }
    if signature not in allowed_signatures_by_level.get(level_name, set()):
        return False, f"{level_name} sequence structure {signature} is not allowed."

    return True, ""


def _validate_sequencing_payload(payload, level_name):
    jumbled_sequence = payload.get("jumbled_sequence")
    correct_index_order = payload.get("correct_index_order")

    expected_lengths = {
        "beginner": 3,
        "intermediate": 4,
        "hard": 5,
    }
    expected_length = expected_lengths.get(level_name)
    if not isinstance(jumbled_sequence, list) or expected_length is None:
        return False, "jumbled_sequence is missing or the level could not be identified."
    if len(jumbled_sequence) != expected_length:
        return False, f"jumbled_sequence must contain exactly {expected_length} items."
    if not isinstance(correct_index_order, list) or len(correct_index_order) != expected_length:
        return False, f"correct_index_order must contain exactly {expected_length} entries."

    emojis = []
    size_ratings = []
    for item in jumbled_sequence:
        if not isinstance(item, dict):
            return False, "Every jumbled_sequence entry must be an object."
        emoji = item.get("emoji")
        size_rating = item.get("size_rating")
        if emoji is None or size_rating is None:
            return False, "Each sequencing item must include emoji and size_rating."
        if not isinstance(emoji, str) or not emoji.strip():
            return False, "Each sequencing item must include a non-empty emoji."
        emojis.append(emoji.strip())
        if not isinstance(size_rating, int):
            return False, "Each sequencing item must include an integer size_rating."
        size_ratings.append(size_rating)

    if len(set(emojis)) != 1:
        return False, "All sequencing items should use the same emoji."
    if len(set(size_ratings)) != expected_length:
        return False, "size_rating values must be unique per item."

    sorted_ratings = sorted(size_ratings)
    rank_lookup = {rating: rank for rank, rating in enumerate(sorted_ratings, start=1)}
    expected_order = [rank_lookup[rating] for rating in size_ratings]
    if correct_index_order != expected_order:
        return False, f"correct_index_order is wrong. Expected {expected_order} (per-item rank in shown order, 1=smallest)."

    if size_ratings == sorted_ratings:
        return False, "jumbled_sequence must be jumbled; do not return items already in smallest-to-biggest order."

    label_rank = {"tiny": 1, "small": 2, "medium": 3, "large": 4, "huge": 5}
    for item in jumbled_sequence:
        label = item.get("label")
        if isinstance(label, str):
            mapped = label_rank.get(label.strip().lower())
            if mapped is not None and mapped != item.get("size_rating"):
                return False, "When labels are present, each label must match its size_rating."

    return True, ""


def _repair_sequencing_payload(payload, level_name):
    if not isinstance(payload, dict):
        return None

    expected_lengths = {"beginner": 3, "intermediate": 4, "hard": 5}
    expected_length = expected_lengths.get(level_name)
    if expected_length is None:
        return None

    jumbled_sequence = payload.get("jumbled_sequence")
    if not isinstance(jumbled_sequence, list) or not jumbled_sequence:
        return None

    theme = str(payload.get("theme", "")).strip().lower()
    theme_fallbacks = {
        "animals": "Ã°Å¸ÂÂ»",
        "toy": "Ã°Å¸Å¡â€š",
        "toys": "Ã°Å¸Å¡â€š",
        "food": "Ã°Å¸ÂÅ½",
        "shapes": "Ã¢Â­Â",
        "nature": "Ã°Å¸Å’Â³",
    }
    fallback_emoji = "Ã¢Â­Â"
    for key, emoji in theme_fallbacks.items():
        if key in theme:
            fallback_emoji = emoji
            break

    source_emoji = None
    for item in jumbled_sequence:
        if isinstance(item, dict):
            emoji = item.get("emoji")
            if isinstance(emoji, str) and emoji.strip():
                source_emoji = emoji.strip()
                break
    base_emoji = source_emoji or fallback_emoji

    rating_orders = {
        "beginner": [[3, 1, 5], [5, 3, 1], [1, 5, 3]],
        "intermediate": [[2, 4, 1, 3], [3, 1, 4, 2], [4, 2, 1, 3]],
        "hard": [[2, 5, 1, 4, 3], [4, 1, 5, 3, 2], [3, 5, 2, 1, 4]],
    }
    label_lookup = {1: "Tiny", 2: "Small", 3: "Medium", 4: "Large", 5: "Huge"}
    level_orders = rating_orders.get(level_name)
    if not level_orders:
        return None

    theme_bucket = sum(ord(char) for char in theme) % len(level_orders)
    rating_order = level_orders[theme_bucket]

    repaired_items = []
    for index in range(expected_length):
        rating = rating_order[index]
        repaired_items.append({"emoji": base_emoji, "label": label_lookup[rating], "size_rating": rating})

    repaired_payload = dict(payload)
    repaired_payload["jumbled_sequence"] = repaired_items
    size_ratings = [item["size_rating"] for item in repaired_items]
    sorted_ratings = sorted(size_ratings)
    rank_lookup = {rating: rank for rank, rating in enumerate(sorted_ratings, start=1)}
    repaired_payload["correct_index_order"] = [rank_lookup[rating] for rating in size_ratings]
    return repaired_payload


PATTERN_ALLOWED_FAMILIES = {
    "beginner": "ABAB, AABB",
    "intermediate": "AABAAB, ABBABB, ABCABC, AABBAA",
    "hard": "ABACABAC, ABCABCAB, AABCAABC, ABBCABBC",
}

PATTERN_ALLOWED_SIGNATURES = {
    "beginner": ["ABABA", "AABBA"],
    "intermediate": ["AABAAB", "ABBABB", "ABCABC", "AABBAA"],
    "hard": ["ABACABAC", "ABCABCAB", "AABCAABC", "ABBCABBC"],
}
_PATTERN_FAMILY_DECKS = {
    "beginner": [],
    "intermediate": [],
    "hard": [],
}


def _next_pattern_target_signature(level_name):
    allowed = list(PATTERN_ALLOWED_SIGNATURES.get(level_name, []))
    if not allowed:
        return None

    deck = _PATTERN_FAMILY_DECKS.get(level_name)
    if deck is None:
        deck = []
        _PATTERN_FAMILY_DECKS[level_name] = deck

    if not deck:
        deck.extend(allowed)
        random.SystemRandom().shuffle(deck)

    return deck.pop(0)


def _coerce_pattern_payload_to_signature(payload, level_name, target_signature):
    if not isinstance(payload, dict) or not target_signature:
        return payload

    sequence = payload.get("sequence")
    expected_lengths = {"beginner": 5, "intermediate": 6, "hard": 8}
    expected_length = expected_lengths.get(level_name)
    if not isinstance(sequence, list) or expected_length is None:
        return payload
    if len(sequence) != expected_length or len(target_signature) != expected_length:
        return payload

    cleaned = []
    for item in sequence:
        if not isinstance(item, str) or not item.strip():
            return payload
        token = item.strip()
        if token == "?":
            return payload
        cleaned.append(token)

    needed_unique = len({char for char in target_signature})
    tokens = []
    for token in cleaned:
        if token not in tokens:
            tokens.append(token)

    fallback_tokens = ["star", "moon", "tree", "kite", "apple", "cup", "robot", "flower"]
    for token in fallback_tokens:
        if len(tokens) >= needed_unique:
            break
        if token not in tokens:
            tokens.append(token)

    if len(tokens) < needed_unique:
        return payload

    ordered_letters = []
    for char in target_signature:
        if char not in ordered_letters:
            ordered_letters.append(char)

    letter_to_token = {
        letter: tokens[index]
        for index, letter in enumerate(ordered_letters)
        if index < len(tokens)
    }
    if len(letter_to_token) != needed_unique:
        return payload

    rebuilt_sequence = [letter_to_token[char] for char in target_signature]
    updated_payload = dict(payload)
    updated_payload["sequence"] = rebuilt_sequence
    return updated_payload


def _retry_guidance(activity_kind, level_name):
    if activity_kind == "logic":
        return "Reminder: keep the diff_color clearly visible against the specific grid color at diff_index. Compare only to that replaced color."
    if activity_kind == "pattern":
        expected_lengths = {"beginner": 5, "intermediate": 6, "hard": 8}
        expected_length = expected_lengths.get(level_name, "the correct length")
        families = PATTERN_ALLOWED_FAMILIES.get(level_name, PATTERN_ALLOWED_FAMILIES["beginner"])
        return (
            f"Reminder: this {level_name} pattern must contain exactly {expected_length} items in sequence, "
            "with NO question mark and NO blank in the model output. "
            "Do not change the required length. Return only the full sequence. "
            f"Allowed families: {families}."
        )
    if activity_kind == "sequencing":
        return (
            f"Reminder: this {level_name} sequencing puzzle must contain the correct number of items for the level, "
            "use the same emoji in every item, keep the shown sequence jumbled (not already smallest-to-biggest), "
            "and compute correct_index_order as per-item ranks in shown order (1=smallest, N=largest). "
            "Do not enforce a fixed size_rating set for the level."
        )
    return f"Reminder: fix the issue and keep the JSON suitable for the requested activity."


def _build_gemini_judge_prompt(activity_kind, level_name, payload):
    if activity_kind == 'pattern':
        if level_name == 'beginner':
            criteria = "Accept only if replacing ? with correct_answer yields one of these 5-item forms exactly: ABABA or AABBA (letter-consistent mapping). Reject everything else."
        elif level_name == 'intermediate':
            criteria = "Accept only if replacing ? with correct_answer yields one of these 6-item forms exactly: AABAAB, ABBABB, ABCABC, or AABBAA (letter-consistent mapping). Reject everything else."
        else:
            criteria = "Accept only if replacing ? with correct_answer yields one of these 8-item forms exactly: ABACABAC, ABCABCAB, AABCAABC, or ABBCABBC (letter-consistent mapping). Reject everything else."
    elif activity_kind == 'sequencing':
        criteria = 'Only reject if different emojis are mixed, size_rating values are not uniquely rankable, the sequence is already in smallest-to-biggest order (not jumbled), or correct_index_order does not match per-item ranks in shown order (1=smallest, N=largest) computed from the actual size_rating list each time. Example: [3,5,1] -> [2,3,1].'
    else:
        criteria = 'Only reject if diff_index is invalid, or if diff_color is almost indistinguishable from grid_colors[diff_index] at a preschool glance. Compare diff_color only with the replaced slot color, not all grid colors.'

    return f'''You are a strict but fair evaluator for preschool JSON activities.

Activity type: {activity_kind}
Difficulty: {level_name}
Criteria: {criteria}

Important evaluation context:
- Preschoolers do NOT read JSON keys or text. They only see the rendered PNG activity.
- Judge based on whether the rendered task is visually clear and age-appropriate.
- For pattern, enforce letter mapping strictly: same letter means same item string, different letters mean different item strings.
- For logic tasks, compare diff_color only against grid_colors[diff_index] (the replaced slot color).
- For sequencing, compute expected correct_index_order as per-item ranks in shown order (1=smallest, N=largest), using the actual size_rating values in each candidate.
- Method: sort size_rating ascending, assign ranks 1..N, then write each item's rank in the original shown order.
- Example only: if size_rating is [3, 5, 1], expected correct_index_order is [2, 3, 1].

Candidate JSON:
{json.dumps(payload, ensure_ascii=False)}

Return raw JSON only in this format:
{{"suitable": true, "reason": "short reason"}}
or
{{"suitable": false, "reason": "short reason"}}'''



def _judge_payload_with_gemini(activity_kind, level_name, payload):
    if activity_kind == 'sequencing':
        seq_ok, seq_reason = _validate_sequencing_payload(payload, level_name)
        if not seq_ok:
            return {'suitable': False, 'reason': seq_reason}

    judge_prompt = _build_gemini_judge_prompt(activity_kind, level_name, payload)
    return call_google_json(judge_prompt, model=GOOGLE_MODEL, temperature=0)


def _materialize_pattern_answer(payload, level_name):
    if not isinstance(payload, dict):
        return payload

    sequence = payload.get("sequence")
    if not isinstance(sequence, list):
        return payload

    expected_lengths = {"beginner": 5, "intermediate": 6, "hard": 8}
    expected_length = expected_lengths.get(level_name)
    if expected_length is None or len(sequence) != expected_length:
        return payload

    if any(not isinstance(item, str) or not item.strip() for item in sequence):
        return payload

    if any(str(item).strip() == "?" for item in sequence):
        return payload

    full_sequence = [str(item).strip() for item in sequence]
    correct_answer = full_sequence[-1]
    blanked_sequence = full_sequence[:-1] + ["?"]

    materialized = dict(payload)
    materialized["sequence"] = blanked_sequence
    materialized["correct_answer"] = correct_answer
    return materialized


def _generate_openai_payload_core(prompt, filename, folder="activities/json_outputs", model="gpt-4o-mini", temperature=0.8, max_attempts=8):
    activity_kind = _infer_activity_kind(filename)
    level_name = _infer_level(filename)
    working_prompt = prompt
    last_error = None
    target_signature = None

    if activity_kind == "pattern":
        target_signature = _next_pattern_target_signature(level_name)
        if target_signature:
            working_prompt = (
                f"{prompt}\n\n"
                f"STRUCTURE TARGET (must follow exactly): {target_signature}. "
                "Use exactly this letter pattern for this output with one-word items from a single allowed category."
            )

    validators = {
        "logic": _validate_logic_payload,
        "pattern": _validate_pattern_payload,
        "sequencing": _validate_sequencing_payload,
    }
    repairers = {
        "sequencing": _repair_sequencing_payload,
    }

    for attempt in range(1, max_attempts + 1):
        try:
            varied_prompt = _build_generation_prompt(activity_kind, level_name, working_prompt, attempt - 1)
            varied_temperature = min(1.2, temperature + 0.05 * (attempt - 1))
            payload = call_openai_json(varied_prompt, model=model, temperature=varied_temperature)

            if activity_kind == "pattern":
                payload = _coerce_pattern_payload_to_signature(payload, level_name, target_signature)
                payload = _materialize_pattern_answer(payload, level_name)

            validator = validators.get(activity_kind)
            repairer = repairers.get(activity_kind)
            if validator is not None:
                local_ok, local_reason = validator(payload, level_name)
                if not local_ok and repairer is not None:
                    repaired_payload = repairer(payload, level_name)
                    if repaired_payload is not None:
                        payload = repaired_payload
                        local_ok, local_reason = validator(payload, level_name)
                if not local_ok:
                    print(f"Local validation rejected {filename} on attempt {attempt}: {local_reason}")
                    guidance = _retry_guidance(activity_kind, level_name)
                    target_hint = f" Required structure: {target_signature}." if activity_kind == "pattern" and target_signature else ""
                    working_prompt = f"{prompt}\n\nPrevious attempt failed deterministic checks: {local_reason}.{target_hint} {guidance}"
                    continue

            judge_result = _judge_payload_with_gemini(activity_kind, level_name, payload)
            suitable = False
            if isinstance(judge_result, dict):
                suitable_value = judge_result.get("suitable")
                if isinstance(suitable_value, str):
                    suitable = suitable_value.strip().lower() in {"true", "1", "yes"}
                else:
                    suitable = bool(suitable_value)
            judge_reason = "No reason provided by judge."
            if isinstance(judge_result, dict):
                judge_reason = str(judge_result.get("reason", judge_reason))

            if suitable:
                print(f"OpenAI response for {filename}: {json.dumps(payload, ensure_ascii=False)}")
                if activity_kind == "pattern":
                    full_sequence = [payload.get("correct_answer") if item == "?" else item for item in payload.get("sequence", [])]
                    print(f"Pattern target for {filename}: {target_signature}; produced sequence: {full_sequence}")
                print(f"Judge accepted {filename} on attempt {attempt}: {judge_reason}")
                save_json_output(folder, filename, payload)
                return payload

            print(f"Judge rejected {filename} on attempt {attempt}: {judge_reason}")
            working_prompt = f"{prompt}\n\nPrevious attempt was rejected by LLM judge because: {judge_reason}. Generate a different JSON that fixes the issue while keeping the same activity and difficulty."
        except Exception as exc:
            last_error = exc
            print(f"Attempt {attempt} failed for {filename}: {exc}")

    raise RuntimeError(f"Unable to generate an LLM-accepted payload for {filename} after {max_attempts} attempts.") from last_error


In [339]:
_ORIGINAL_GENERATE_OPENAI_PAYLOAD = _generate_openai_payload_core


def _fallback_sequencing_payload(filename, prompt):
    lower_text = f"{filename} {prompt}".lower()
    if "hard" in lower_text:
        size_ratings = [4, 1, 5, 3, 2]
    elif "intermediate" in lower_text:
        size_ratings = [2, 4, 1, 3]
    else:
        size_ratings = [3, 1, 5]

    if "toy" in lower_text or "train" in lower_text:
        theme = "toys"
        emoji = "Ã°Å¸Å¡â€š"
    elif "food" in lower_text or "apple" in lower_text:
        theme = "food"
        emoji = "Ã°Å¸ÂÅ½"
    elif "shape" in lower_text or "circle" in lower_text:
        theme = "shapes"
        emoji = "Ã¢Â­Â"
    elif "nature" in lower_text or "tree" in lower_text:
        theme = "nature"
        emoji = "Ã°Å¸Å’Â³"
    else:
        theme = "animals"
        emoji = "Ã°Å¸ÂÂ»"

    label_lookup = {1: "Tiny", 2: "Small", 3: "Medium", 4: "Large", 5: "Huge"}
    jumbled_sequence = [
        {"emoji": emoji, "label": label_lookup[rating], "size_rating": rating}
        for rating in size_ratings
    ]
    sorted_ratings = sorted(size_ratings)
    rank_lookup = {rating: rank for rank, rating in enumerate(sorted_ratings, start=1)}
    correct_index_order = [rank_lookup[rating] for rating in size_ratings]

    return {
        "theme": theme,
        "jumbled_sequence": jumbled_sequence,
        "correct_index_order": correct_index_order,
    }


def _fallback_logic_payload(filename, prompt):
    lower_text = f"{filename} {prompt}".lower()
    if "hard" in lower_text:
        grid_colors = ["#3B82F6", "#2563EB", "#1D4ED8", "#60A5FA", "#93C5FD", "#0F172A", "#1E3A8A", "#DBEAFE", "#3F83F8"]
        diff_index = 4
        diff_color = "#BFDBFE"
    elif "intermediate" in lower_text:
        grid_colors = ["#FFB3BA", "#FF677D", "#D4A5A5", "#392F5A", "#FFC3A0", "#A7C957", "#FF6F61", "#6B4226", "#FFE156"]
        diff_index = 2
        diff_color = "#6A0572"
    else:
        grid_colors = ["#FF5733", "#33FF57", "#3357FF", "#F1C40F"]
        diff_index = 2
        diff_color = "#FF33A8"

    return {
        "grid_colors": grid_colors,
        "diff_index": diff_index,
        "diff_color": diff_color,
    }


def generate_openai_payload(prompt, filename, folder="activities/json_outputs", model="gpt-4o-mini", temperature=0.8, max_attempts=8):
    try:
        return _ORIGINAL_GENERATE_OPENAI_PAYLOAD(
            prompt,
            filename,
            folder=folder,
            model=model,
            temperature=temperature,
            max_attempts=max_attempts,
        )
    except RuntimeError as exc:
        activity_kind = _infer_activity_kind(filename)
        if activity_kind == "sequencing":
            payload = _fallback_sequencing_payload(filename, prompt)
        elif activity_kind == "logic":
            payload = _fallback_logic_payload(filename, prompt)
        else:
            raise

        print(f"Fallback used for {filename} after generator failure: {exc}")
        print(f"Fallback payload for {filename}: {json.dumps(payload, ensure_ascii=False)}")
        save_json_output(folder, filename, payload)
        return payload


## Pattern Rendering Utilities
Helper functions for loading icons and drawing pattern images.

In [340]:
import requests
from io import BytesIO
import cairosvg
from PIL import Image, ImageDraw, ImageFont
import os
import random
import re

# HELPER FUNCTION 1: Generate a random pastel color
def random_pastel():
    return (
        random.randint(180, 255),
        random.randint(180, 255),
        random.randint(180, 255),
    )

# HELPER FUNCTION 2: Robust get_icon that prefers -fill, tries bi + mdi, injects color when needed
def get_icon(name, size=120, color_hex="333333"):
    """
    Downloads an icon from Iconify and returns it as a colored PNG image.
    Preference order for variants: prefer "name-fill" then "name".
    Tries Bootstrap Icons (bi) then Material Design Icons (mdi).
    If the API doesn't honor the color parameter, fetches the SVG and injects the color.
    """
    name = name.strip().lower()
    color_hex = color_hex.lstrip('#')
    color_with_hash = f"#{color_hex}"
    icon_sets = ["bi", "mdi"]

    # Prefer the '-fill' variant first when possible
    if name.endswith('-fill'):
        name_variants = [name, name[:-5]]
    else:
        name_variants = [f"{name}-fill", name]

    for variant in name_variants:
        for icon_set in icon_sets:
            # 1) Try API with color parameter
            url_col = f"https://api.iconify.design/{icon_set}/{variant}.svg?color=%23{color_hex}"
            try:
                r = requests.get(url_col, timeout=5)
                if r.ok:
                    try:
                        png_bytes = cairosvg.svg2png(bytestring=r.content, output_width=size, output_height=size)
                        return Image.open(BytesIO(png_bytes)).convert("RGBA")
                    except Exception:
                        # conversion failed for colored response; fall back to svg text handling
                        svg_text = r.text
                else:
                    svg_text = None
            except Exception:
                svg_text = None

            # 2) Try API without color parameter and inject color into SVG text
            url_plain = f"https://api.iconify.design/{icon_set}/{variant}.svg"
            try:
                r2 = requests.get(url_plain, timeout=5)
                if r2.ok:
                    svg = r2.text
                    # Replace 'currentColor' with explicit hex color
                    svg = svg.replace('currentColor', color_with_hash)
                    # Fix fills like fill="FF0000" -> fill="#FF0000"
                    svg = re.sub(r'fill=\"([0-9A-Fa-f]{3,6})\"', lambda m: f'fill=\"#{m.group(1)}\"', svg)
                    try:
                        png_bytes = cairosvg.svg2png(bytestring=svg.encode('utf-8'), output_width=size, output_height=size)
                        return Image.open(BytesIO(png_bytes)).convert("RGBA")
                    except Exception:
                        pass
            except Exception:
                pass
    # If all attempts fail
    print(f"Failed to load {name} from any icon set or variant")
    return None

# HELPER FUNCTION 3: Generate a random hex color
def random_hex_color():
    return f"{random.randint(0,255):02X}{random.randint(0,255):02X}{random.randint(0,255):02X}"

# MAIN FUNCTION: Create pattern images
def generate_pattern_image(json_input):
    sequence = json_input["sequence"]
    card_size = 200
    spacing = 20
    total_width = (card_size * len(sequence)) + (spacing * (len(sequence) + 1))
    total_height = card_size + (spacing * 2)

    # Use a solid white background instead of a random gradient
    canvas = Image.new("RGB", (total_width, total_height), color='white')
    draw = ImageDraw.Draw(canvas)

    icon_colors = {}
    x = spacing
    for item in sequence:
        draw.rectangle([x, spacing, x + card_size, spacing + card_size], fill="white", outline="black", width=3)
        if item != "?":
            if item not in icon_colors:
                icon_colors[item] = random_hex_color()
            icon_color = icon_colors[item]
            icon = get_icon(item, color_hex=icon_color)
            if icon:
                ix = int(x + (card_size - icon.width) / 2)
                iy = int(spacing + (card_size - icon.height) / 2)
                canvas.paste(icon, (ix, iy), icon)
        x += card_size + spacing

    if not os.path.exists('activities/pattern'):
        os.makedirs('activities/pattern')

    file_number = 1
    output_path = f"activities/pattern/pattern_activity_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/pattern/pattern_activity_{file_number}.png"

    canvas.save(output_path)
    print(f"Full pattern image created: {output_path}")
    return output_path

# quick test
#test_data = {"sequence":["moon","moon","sun","star","moon","moon","sun","?"]}
#generate_pattern_image(test_data)


## Shared Worksheet Builder
Used by logic, pattern, and sequencing worksheet pages.

In [341]:
from PIL import Image, ImageDraw, ImageFont
import os


def load_font(size, bold=False):
    font_paths = [
        r"C:\Windows\Fonts\arialbd.ttf" if bold else r"C:\Windows\Fonts\arial.ttf",
        r"C:\Windows\Fonts\segoeuib.ttf" if bold else r"C:\Windows\Fonts\segoeui.ttf",
    ]
    for font_path in font_paths:
        if os.path.exists(font_path):
            try:
                return ImageFont.truetype(font_path, size=size)
            except Exception:
                pass
    return ImageFont.load_default()


# Create a workbook-style page with 3 copies of an activity
def create_printable_worksheet(activity_func, test_jsons, activity_name="worksheet", level_label="LEVEL", instruction_text="", question_prompt="Circle the different answer.", out_dir='activities/worksheets', spacing=18):
    os.makedirs(out_dir, exist_ok=True)

    activity_paths = []
    for test_json in test_jsons:
        path = None
        if isinstance(test_json, dict):
            cached_path = test_json.get("__image_path")
            if isinstance(cached_path, str) and cached_path and os.path.exists(cached_path):
                path = cached_path

        if path is None:
            path = activity_func(test_json)
            if path and isinstance(test_json, dict):
                test_json["__image_path"] = path

        if path:
            activity_paths.append(path)

    if not activity_paths:
        print("No activities generated!")
        return None

    max_activity_width = 1300
    imgs = []
    for path in activity_paths:
        im = Image.open(path).convert('RGBA')
        if im.width > max_activity_width:
            new_height = int(im.height * (max_activity_width / im.width))
            im = im.resize((max_activity_width, new_height), Image.Resampling.LANCZOS)
        imgs.append(im)

    page_width = 1900
    margin_x = 100
    header_height = 260
    prompt_height = 60
    question_pad = 20

    page_height = header_height + sum(im.height + prompt_height + (question_pad * 2) for im in imgs) + spacing * (len(imgs) - 1) + 60
    canvas = Image.new('RGBA', (page_width, page_height), (255, 255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    title_font = load_font(50, bold=True)
    label_font = load_font(32, bold=True)
    body_font = load_font(30, bold=False)
    prompt_font = load_font(32, bold=True)

    draw.rectangle([0, 0, page_width, header_height], fill=(255, 255, 255, 255))

    title = activity_name.replace('_', ' ').upper()
    draw.text((margin_x, 24), title, fill=(25, 25, 25), font=title_font, anchor='la')
    draw.text((margin_x, 92), f"Level: {level_label}", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.text((margin_x + 380, 92), "Name: ____________________________", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.text((margin_x + 1040, 92), "Date: __________________", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.multiline_text((margin_x, 148), instruction_text or "Instructions: Look carefully at each set and circle the one that is different.", fill=(45, 45, 45), font=body_font, spacing=5)
    draw.text((margin_x, 202), "Complete all 3 questions on this page.", fill=(45, 45, 45), font=body_font, anchor='la')

    y = header_height + 22
    for index, im in enumerate(imgs, start=1):
        block_top = y
        block_bottom = y + prompt_height + im.height + (question_pad * 2)

        draw.rounded_rectangle([margin_x - 14, block_top, page_width - margin_x + 14, block_bottom], radius=22, outline=(210, 210, 210), width=2, fill=(255, 255, 255, 255))

        prompt = f"Question {index}. {question_prompt}"
        draw.text((margin_x + 16, block_top + 16), prompt, fill=(30, 30, 30), font=prompt_font, anchor='la')

        image_y = block_top + prompt_height + question_pad
        image_x = (page_width - im.width) // 2
        canvas.paste(im, (image_x, image_y), im)

        y = block_bottom + spacing

    out_path = os.path.join(out_dir, f'{activity_name}.png')
    canvas.save(out_path)
    print(f"Worksheet saved: {out_path}")
    print(f"Worksheet size: {canvas.size[0]}x{canvas.size[1]}")
    return out_path

## Pattern Prompts and Worksheet Generation
Pattern prompt templates, generation calls, and worksheet rendering.

In [342]:
pattern_beginner_base = """BEGINNER:

GLOBAL ITEM POOL (must be used across all generations, but NEVER repeat items already used earlier in the conversation):

shapes: circle, triangle, square, heart, star, octagon, pentagon, sun, moon
animals: cat, dog, duck, pig, cow, bee, owl, bat, sheep, rabbit, alien, butterfly, elephant
objects: chair, cup, clock, book, pencil, bag, bell, brush, key, lamp, dollar, basketball, balloon, eye
toys: robot, train, car, kite, truck, puzzle, airplane, scooter
nature: leaf, tree, cloud, water, flower, snowflake,
food: apple, bread, carrot, cookie, cheese, egg, corn, cake, baguette, cupcake

RULE: use items in same category, and time after time can mix around.
ALLOWED CATEGORIES (must all be available for selection across generations): shapes, animals, objects, toys, nature, food.
DIVERSITY RULE: Do NOT use cat or dog for pattern items.
ALLOWED STRUCTURES FOR BEGINNER: ABAB or AABB or AAAA only.

you are a JSON generator. create a pattern and repetition (5 items). use only 2 easy preschool items, choose at random. the pattern must be very simple and obvious (ABAB or AABB style). output ONLY raw JSON in this format: {\"sequence\": []}. do not include any conversational text. use 1-word items. make sure each generation uses only the word bank above and stays within one category. do not use colours. IMPORTANT: return the COMPLETE sequence with no blank and no '?'.
"""

pattern_intermediate_base = """INTERMEDIATE:

GLOBAL ITEM POOL (must be used across all generations, but NEVER repeat items already used earlier in the conversation):

shapes: circle, triangle, square, heart, star, octagon, pentagon, sun, moon
animals: cat, dog, duck, pig, cow, bee, owl, bat, sheep, rabbit, alien, butterfly, elephant
objects: chair, cup, clock, book, pencil, bag, bell, brush, key, lamp, dollar, basketball, balloon, eye
toys: robot, train, car, kite, truck, puzzle, airplane, scooter
nature: leaf, tree, cloud, water, flower, snowflake,
food: apple, bread, carrot, cookie, cheese, egg, corn, cake, baguette, cupcake

RULE: use items in same category, and time after time can mix around.
ALLOWED CATEGORIES (must all be available for selection across generations): shapes, animals, objects, toys, nature, food.
DIVERSITY RULE: Do NOT use cat or dog for pattern items.
ALLOWED STRUCTURES FOR INTERMEDIATE: AABAAB or ABBABB or ABCABC or AABBAA only.

you are a JSON generator. create a pattern and repetition (6 items). use 2 or 3 different easy preschool items, choose at random. the pattern must follow one of the allowed structures listed above exactly. output ONLY raw JSON in this format: {\"sequence\": []}. do not include any conversational text. use 1-word items. each generation must stay inside the word bank above and use one category only. do not use colours. IMPORTANT: return the COMPLETE sequence with no blank and no '?'.
"""

pattern_hard_base = """HARD:

GLOBAL ITEM POOL (must be used across all generations, but NEVER repeat items already used earlier in the conversation):

shapes: circle, triangle, square, heart, star, octagon, pentagon, sun, moon
animals: cat, dog, duck, pig, cow, bee, owl, bat, sheep, rabbit, alien, butterfly, elephant
objects: chair, cup, clock, book, pencil, bag, bell, brush, key, lamp, dollar, basketball, balloon, eye
toys: robot, train, car, kite, truck, puzzle, airplane, scooter
nature: leaf, tree, cloud, water, flower, snowflake,
food: apple, bread, carrot, cookie, cheese, egg, corn, cake, baguette, cupcake

RULE: use items in same category, and time after time can mix around.
ALLOWED CATEGORIES (must all be available for selection across generations): shapes, animals, objects, toys, nature, food.
DIVERSITY RULE: Do NOT use cat or dog for pattern items.
ALLOWED STRUCTURES FOR HARD: ABACABAC, ABCABCAB, AABCAABC, or ABBCABBC only.

you are a JSON generator. create a pattern and repetition (8 items). use 3 easy preschool items, choose at random. the pattern must follow a hidden structure such as overlapping cycles, alternating dual-rules, or delayed repetition (e.g. ABACABAC, ABCABCAB, AABCAABC, ABBCABBC, ACABACAB, or similar structured but non-obvious logic). output ONLY raw JSON in this format: {\"sequence\": []}. do not include any conversational text. use 1-word items. each generation must stay inside the word bank above and use one category only. do not use colours. IMPORTANT: return the COMPLETE sequence with no blank and no '?'.
HARD UNIQUENESS RULE: prefer structures like ABACABAC, ABCABCAB, AABCAABC, ABBCABBC, ACABACAB, or other non-trivial but valid 8-item logic patterns using 3 unique items.
"""

def _make_pattern_prompt(base_prompt, theme):
    return f"{base_prompt} {theme}"


def _pattern_items_from_sequence(sequence):
    if not isinstance(sequence, list):
        return set()
    return {
        str(item).strip().lower()
        for item in sequence
        if isinstance(item, str) and item.strip() and item.strip() != "?"
    }


def _pattern_signature(sequence):
    if not isinstance(sequence, list) or not sequence:
        return None
    return "|".join(str(item).strip().lower() for item in sequence)


def _load_existing_pattern_history(folder="activities/json_outputs"):
    history = {
        "beginner": {"items": set(), "signatures": set()},
        "intermediate": {"items": set(), "signatures": set()},
        "hard": {"items": set(), "signatures": set()},
    }

    folder_path = Path(folder)
    if not folder_path.exists():
        return history

    for json_path in folder_path.glob("pattern_*.json"):
        level_name = _infer_level(json_path.name)
        level_history = history.get(level_name)
        if level_history is None:
            continue

        try:
            with json_path.open("r", encoding="utf-8") as fh:
                payload = json.load(fh)
        except Exception:
            continue

        sequence = payload.get("sequence") if isinstance(payload, dict) else None
        signature = _pattern_signature(sequence)
        if signature:
            level_history["signatures"].add(signature)
        level_history["items"].update(_pattern_items_from_sequence(sequence))

    return history


def _augment_pattern_prompt_for_regeneration(prompt_name, banned_items=None, banned_signatures=None):
    meta = PATTERN_PROMPT_META[prompt_name]
    prompt_text = _make_pattern_prompt(meta["base_prompt"], meta["theme"])

    constraints = [
        "",
        "REGENERATION CONSTRAINTS (must follow exactly):",
        "1) Return only full sequence JSON: {\"sequence\": []}.",
        "2) Do NOT include '?' and do NOT include correct_answer.",
        "3) Do not copy any exact sequence from previous generations.",
        "4) Choose any one allowed category RANDOMLY for this output. Do not use a fixed category per file.",
        "5) Pick items that are far apart in the chosen category list (not adjacent). Prefer index distance >= 3.",
        "6) Avoid always taking the first few words in the category list.",
    ]

    if banned_items:
        banned_items_text = ", ".join(sorted({str(item).strip().lower() for item in banned_items if str(item).strip()}))
        constraints.append(f"7) DO NOT use these previously used items for this level: {banned_items_text}.")
    else:
        constraints.append("7) Use fresh items not used in very recent outputs for this level.")

    if banned_signatures:
        signatures_preview = "; ".join(sorted(banned_signatures)[:12])
        constraints.append(f"8) BANNED exact sequence signatures: {signatures_preview}.")
    else:
        constraints.append("8) Avoid repeating exact prior sequence ordering.")

    constraints.append("9) Keep one category only and use simple one-word items from the allowed pool.")

    return prompt_text + "\n" + "\n".join(constraints)


pattern_prompt_specs = [
    ("pattern_beginner_prompt_1", pattern_beginner_base, "Choose any one allowed category randomly.", "beginner"),
    ("pattern_beginner_prompt_2", pattern_beginner_base, "Choose any one allowed category randomly.", "beginner"),
    ("pattern_beginner_prompt_3", pattern_beginner_base, "Choose any one allowed category randomly.", "beginner"),
    ("pattern_intermediate_prompt_1", pattern_intermediate_base, "Choose any one allowed category randomly.", "intermediate"),
    ("pattern_intermediate_prompt_2", pattern_intermediate_base, "Choose any one allowed category randomly.", "intermediate"),
    ("pattern_intermediate_prompt_3", pattern_intermediate_base, "Choose any one allowed category randomly.", "intermediate"),
    ("pattern_hard_prompt_1", pattern_hard_base, "Choose any one allowed category randomly.", "hard"),
    ("pattern_hard_prompt_2", pattern_hard_base, "Choose any one allowed category randomly.", "hard"),
    ("pattern_hard_prompt_3", pattern_hard_base, "Choose any one allowed category randomly.", "hard"),
]

PATTERN_PROMPT_META = {}
for prompt_name, base_prompt, theme, level_name in pattern_prompt_specs:
    globals()[prompt_name] = _make_pattern_prompt(base_prompt, theme)
    PATTERN_PROMPT_META[prompt_name] = {
        "base_prompt": base_prompt,
        "theme": theme,
        "level": level_name,
    }

In [343]:
# Duplicate pattern prompt block intentionally disabled.
# Active prompt definitions live in the earlier Pattern Prompts and Worksheet Generation cell.
pass

In [344]:
SELECTED_ACTIVITY = None
SELECTED_DIFFICULTY = None


def set_selected_worksheet(activity, difficulty):
    global SELECTED_ACTIVITY, SELECTED_DIFFICULTY
    SELECTED_ACTIVITY = activity
    SELECTED_DIFFICULTY = difficulty


def _prompt_activity_and_level(prompt):
    lower_prompt = prompt.lower()

    if "jumbled_sequence" in lower_prompt or "size_rating" in lower_prompt or "size-sequencing" in lower_prompt:
        activity_kind = "sequencing"
    elif "grid_colors" in lower_prompt or "spot the difference" in lower_prompt:
        activity_kind = "logic"
    else:
        activity_kind = "pattern"

    if "beginner" in lower_prompt:
        level_name = "beginner"
    elif "intermediate" in lower_prompt:
        level_name = "intermediate"
    elif "hard" in lower_prompt:
        level_name = "hard"
    else:
        level_name = "unknown"

    return activity_kind, level_name

In [345]:
_original_generate_openai_payload = generate_openai_payload
_original_save_json_output = save_json_output
_original_generate_pattern_image = generate_pattern_image


def _lazy_payload():
    return {"__skip__": True}


def generate_openai_payload(prompt, filename, folder="activities/json_outputs", model="gpt-4o-mini", temperature=0.8, max_attempts=8):
    selected_activity = globals().get("SELECTED_ACTIVITY")
    selected_difficulty = globals().get("SELECTED_DIFFICULTY")
    activity_kind, level_name = _prompt_activity_and_level(prompt)

    if selected_activity is None or selected_difficulty is None:
        return _lazy_payload()

    if selected_activity != activity_kind or selected_difficulty != level_name:
        return _lazy_payload()

    return _original_generate_openai_payload(
        prompt,
        filename,
        folder=folder,
        model=model,
        temperature=temperature,
        max_attempts=max_attempts,
    )


def save_json_output(folder, filename, payload):
    if isinstance(payload, dict) and payload.get("__skip__"):
        return None
    return _original_save_json_output(folder, filename, payload)


def generate_pattern_image(json_input):
    if not json_input or (isinstance(json_input, dict) and json_input.get("__skip__")):
        return None
    return _original_generate_pattern_image(json_input)


## Pattern Generation
Generate JSON payloads and preview the pattern images.

In [346]:
pattern_generation_specs = [
    ("pattern_beginner_1", "pattern_beginner_prompt_1", "pattern_beginner_1.json", 0.6),
    ("pattern_beginner_2", "pattern_beginner_prompt_2", "pattern_beginner_2.json", 0.6),
    ("pattern_beginner_3", "pattern_beginner_prompt_3", "pattern_beginner_3.json", 0.6),
    ("pattern_intermediate_1", "pattern_intermediate_prompt_1", "pattern_intermediate_1.json", 0.55),
    ("pattern_intermediate_2", "pattern_intermediate_prompt_2", "pattern_intermediate_2.json", 0.55),
    ("pattern_intermediate_3", "pattern_intermediate_prompt_3", "pattern_intermediate_3.json", 0.55),
    ("pattern_hard_1", "pattern_hard_prompt_1", "pattern_hard_1.json", 0.5),
    ("pattern_hard_2", "pattern_hard_prompt_2", "pattern_hard_2.json", 0.5),
    ("pattern_hard_3", "pattern_hard_prompt_3", "pattern_hard_3.json", 0.5),
]

pattern_history = _load_existing_pattern_history()

pattern_payloads = {}
for variable_name, prompt_name, filename, temperature in pattern_generation_specs:
    level_name = _infer_level(filename)
    level_history = pattern_history.setdefault(level_name, {"items": set(), "signatures": set()})

    prompt_text = _augment_pattern_prompt_for_regeneration(
        prompt_name,
        banned_items=level_history.get("items", set()),
        banned_signatures=level_history.get("signatures", set()),
    )

    payload = generate_openai_payload(prompt_text, filename, temperature=temperature)

    sequence = payload.get("sequence") if isinstance(payload, dict) else None
    level_history["items"].update(_pattern_items_from_sequence(sequence))
    signature = _pattern_signature(sequence)
    if signature:
        level_history["signatures"].add(signature)

    pattern_payloads[variable_name] = payload
    globals()[variable_name] = payload
    image_path = generate_pattern_image(payload)
    if image_path and isinstance(payload, dict):
        payload["__image_path"] = image_path

## Pattern Worksheets
Build the printable worksheet pages from the generated pattern JSON.

In [347]:
def _generate_pattern_worksheet(activity_name, level_label, payload_names):
    return create_printable_worksheet(
        generate_pattern_image,
        [globals()[name] for name in payload_names],
        activity_name=activity_name,
        level_label=level_label,
        instruction_text="Instructions: Fill in the blank in each pattern.",
        question_prompt="Fill in the blank."
    )


def generate_pattern_worksheet():
    return _generate_pattern_worksheet(
        "BEGINNER_PATTERN_RECOGNITION",
        "Beginner",
        ["pattern_beginner_1", "pattern_beginner_2", "pattern_beginner_3"],
    )


def generate_pattern_intermediate_worksheet():
    return _generate_pattern_worksheet(
        "INTERMEDIATE_PATTERN_RECOGNITION",
        "Intermediate",
        ["pattern_intermediate_1", "pattern_intermediate_2", "pattern_intermediate_3"],
    )


def generate_pattern_hard_worksheet():
    return _generate_pattern_worksheet(
        "HARD_PATTERN_RECOGNITION",
        "Hard",
        ["pattern_hard_1", "pattern_hard_2", "pattern_hard_3"],
    )


generate_pattern_worksheet()
generate_pattern_intermediate_worksheet()
generate_pattern_hard_worksheet()

No activities generated!
No activities generated!
No activities generated!


## Logic / Spot the Difference
Logic puzzle prompts, generators, and worksheet rendering.

In [348]:
#BEGINNER
from PIL import Image, ImageDraw
import os
import random

def generate_beginner_spot_diff(llm_json):
    # 1. Settings (2x2 Grid)
    grid_size = 2
    square_size = 180
    gap = 25
    grid_pixel_size = (grid_size * square_size) + ((grid_size + 1) * gap)

    # Optional transparent background flag and neutral panel/border colors
    transparent = llm_json.get('transparent', False)
    panel_color = llm_json.get('panel_color', '#EEEEEE')
    border_color = llm_json.get('border_color', (200, 200, 200))

    margin = 50
    canvas_w = (grid_pixel_size * 2) + (margin * 3)
    canvas_h = grid_pixel_size + (margin * 2)

    if transparent:
        canvas = Image.new('RGBA', (canvas_w, canvas_h), color=(255, 255, 255, 0))
    else:
        canvas = Image.new('RGB', (canvas_w, canvas_h), color='white')
    draw = ImageDraw.Draw(canvas)

    # Prepare colors (expects list of 4)
    original_colors = llm_json['grid_colors']
    diff_index = llm_json['diff_index']
    modified_colors = list(original_colors)
    modified_colors[diff_index] = llm_json['diff_color']

    def draw_grid_at(start_x, color_list):
        panel_box = [start_x, margin - 10, start_x + grid_pixel_size, canvas_h - margin + 10]
        draw.rounded_rectangle(panel_box, radius=18, fill=panel_color)
        border_box = [start_x + 5, margin - 5, start_x + grid_pixel_size - 5, canvas_h - margin + 5]
        draw.rounded_rectangle(border_box, radius=18, outline=border_color, width=5)

        for i, color in enumerate(color_list):
            row = i // grid_size
            col = i % grid_size
            x1 = start_x + gap + (col * (square_size + gap))
            y1 = margin + gap + (row * (square_size + gap))
            x2 = x1 + square_size
            y2 = y1 + square_size
            draw.rectangle([x1, y1, x2, y2], fill=color, outline='white', width=2)

    draw_grid_at(margin, original_colors)
    draw_grid_at(grid_pixel_size + (margin * 2), modified_colors)

    # 5. Save
    if not os.path.exists('activities/logic problem solving'):
        os.makedirs('activities/logic problem solving')

    file_number = 1
    output_path = f"activities/logic problem solving/BEGINNER_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/logic problem solving/BEGINNER_{file_number}.png"

    canvas.save(output_path)
    print(f"Full logic problem image created: {output_path}")
    return output_path

# Test JSON
#beginner_json = {"theme":"Sky Picnic Parade","grid_colors":["#2E8B57","#8B4513","#4682B4","#DAA520"],"diff_index":1,"diff_color":"#6A0DAD"}

#generate_beginner_spot_diff(beginner_json)


In [349]:
#INTERMEDIATE
from PIL import Image, ImageDraw
import os
import random

def generate_spot_the_diff_intermediate(llm_json):
    # 1. Settings
    grid_size = 3
    square_size = 120
    gap = 15
    grid_pixel_size = (grid_size * square_size) + ((grid_size + 1) * gap)

    # Optional transparent background flag and neutral panel/border colors
    transparent = llm_json.get('transparent', False)
    panel_color = llm_json.get('panel_color', '#EEEEEE')
    border_color = llm_json.get('border_color', (200, 200, 200))

    # Canvas setup (Side-by-Side)
    margin = 50
    canvas_w = (grid_pixel_size * 2) + (margin * 3)
    canvas_h = grid_pixel_size + (margin * 2)

    if transparent:
        canvas = Image.new('RGBA', (canvas_w, canvas_h), color=(255, 255, 255, 0))
    else:
        canvas = Image.new('RGB', (canvas_w, canvas_h), color='white')
    draw = ImageDraw.Draw(canvas)

    # 2. Prepare the two color lists
    original_colors = llm_json['grid_colors']
    diff_index = llm_json['diff_index']

    # Create the "modified" list
    modified_colors = list(original_colors)
    modified_colors[diff_index] = llm_json['diff_color']

    # 3. Helper to draw the grid
    def draw_grid_at(start_x, color_list):
        # Draw a neutral panel behind the grid to help the colors pop
        panel_box = [start_x, margin - 10, start_x + grid_pixel_size, canvas_h - margin + 10]
        draw.rounded_rectangle(panel_box, radius=18, fill=panel_color)

        # Draw the neutral border box around the panel
        border_box = [start_x + 5, margin - 5, start_x + grid_pixel_size - 5, canvas_h - margin + 5]
        draw.rounded_rectangle(border_box, radius=18, outline=border_color, width=5)

        for i, color in enumerate(color_list):
            row = i // grid_size
            col = i % grid_size

            x1 = start_x + gap + (col * (square_size + gap))
            y1 = margin + gap + (row * (square_size + gap))
            x2 = x1 + square_size
            y2 = y1 + square_size

            # Draw solid squares instead of circles
            draw.rectangle([x1, y1, x2, y2], fill=color, outline='white', width=2)

    # 4. Render Left and Right
    draw_grid_at(margin, original_colors)
    draw_grid_at(grid_pixel_size + (margin * 2), modified_colors)

    # 5. Save
    if not os.path.exists('activities/logic problem solving'):
        os.makedirs('activities/logic problem solving')

    file_number = 1
    output_path = f"activities/logic problem solving/INTERMEDDIATE_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/logic problem solving/INTERMEDDIATE_{file_number}.png"

    canvas.save(output_path)
    print(f"Full logic problem image created: {output_path}")
    return output_path

# --- TEST DATA ---
#test_json = {"theme":"Desert Mirage Kite Bazaar at Sunset","grid_colors":["#E7C9A3","#3CCFCF","#D85C3A","#FFB45A","#4F8A5B","#C46A3C","#FF7AA2","#7A4B2A","#4DA3FF"],"diff_index":3,"diff_color":"#B46CFF"}
#generate_spot_the_diff_intermediate(test_json)


In [350]:
#HARD
from PIL import Image, ImageDraw
import os
import random

def generate_spot_the_diff_hard(llm_json):
    # 1. Settings
    grid_size = 3
    square_size = 120
    gap = 15
    grid_pixel_size = (grid_size * square_size) + ((grid_size + 1) * gap)

    # Optional transparent background flag and neutral panel/border colors
    transparent = llm_json.get('transparent', False)
    panel_color = llm_json.get('panel_color', '#EEEEEE')
    border_color = llm_json.get('border_color', (200, 200, 200))

    # Canvas setup (Side-by-Side)
    margin = 50
    canvas_w = (grid_pixel_size * 2) + (margin * 3)
    canvas_h = grid_pixel_size + (margin * 2)

    if transparent:
        canvas = Image.new('RGBA', (canvas_w, canvas_h), color=(255, 255, 255, 0))
    else:
        canvas = Image.new('RGB', (canvas_w, canvas_h), color='white')
    draw = ImageDraw.Draw(canvas)

    # 2. Prepare the two color lists
    original_colors = llm_json['grid_colors']
    diff_index = llm_json['diff_index']

    # Create the "modified" list
    modified_colors = list(original_colors)
    modified_colors[diff_index] = llm_json['diff_color']

    # 3. Helper to draw the grid
    def draw_grid_at(start_x, color_list):
        # Draw a neutral panel behind the grid to help the colors pop
        panel_box = [start_x, margin - 10, start_x + grid_pixel_size, canvas_h - margin + 10]
        draw.rounded_rectangle(panel_box, radius=18, fill=panel_color)

        # Draw the neutral border box around the panel
        border_box = [start_x + 5, margin - 5, start_x + grid_pixel_size - 5, canvas_h - margin + 5]
        draw.rounded_rectangle(border_box, radius=18, outline=border_color, width=5)

        for i, color in enumerate(color_list):
            row = i // grid_size
            col = i % grid_size

            x1 = start_x + gap + (col * (square_size + gap))
            y1 = margin + gap + (row * (square_size + gap))
            x2 = x1 + square_size
            y2 = y1 + square_size

            # Draw solid squares instead of circles
            draw.rectangle([x1, y1, x2, y2], fill=color, outline='white', width=2)

    # 4. Render Left and Right
    draw_grid_at(margin, original_colors)
    draw_grid_at(grid_pixel_size + (margin * 2), modified_colors)

    # 5. Save
    if not os.path.exists('activities/logic problem solving'):
        os.makedirs('activities/logic problem solving')

    file_number = 1
    output_path = f"activities/logic problem solving/HARD_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/logic problem solving/HARD_{file_number}.png"

    canvas.save(output_path)
    print(f"Full logic problem image created: {output_path}")
    return output_path

# --- TEST DATA ---
#test_json = {"theme":"Lavender Quantum Dream Observatory","grid_colors":["#6A3FA0","#E6D7FF","#2B0A3D","#C000FF","#9B7BB8","#7A2CFF","#D1B3FF","#3A145A","#B100FF"],"diff_index":4,"diff_color":"#F8E9FF"}

#generate_spot_the_diff_hard(test_json)


In [351]:
_original_generate_beginner_spot_diff = generate_beginner_spot_diff
_original_generate_spot_the_diff_intermediate = generate_spot_the_diff_intermediate
_original_generate_spot_the_diff_hard = generate_spot_the_diff_hard


def _logic_payload_is_skipped(llm_json):
    return not llm_json or (isinstance(llm_json, dict) and llm_json.get("__skip__"))


def generate_beginner_spot_diff(llm_json):
    if _logic_payload_is_skipped(llm_json):
        return None
    return _original_generate_beginner_spot_diff(llm_json)


def generate_spot_the_diff_intermediate(llm_json):
    if _logic_payload_is_skipped(llm_json):
        return None
    return _original_generate_spot_the_diff_intermediate(llm_json)


def generate_spot_the_diff_hard(llm_json):
    if _logic_payload_is_skipped(llm_json):
        return None
    return _original_generate_spot_the_diff_hard(llm_json)


## Logic Worksheets
Build the printable worksheet pages for logic puzzles.

In [352]:
# WORKSHEET GENERATOR FOR TEACHERS - 3 of same activity per sheet
# LOGIC AND PROBLEM SOLVING ACTIVITY

from PIL import Image, ImageDraw, ImageFont
import os
import random


logic_beginner_prompt_1 = '''You are an Activity Generator for 4-year-olds. Create a 'Spot the Difference' game. Instructions: Choose any playful theme (do not lock into the same few themes). Provide grid_colors as a list of 4 child-friendly, clearly distinct Hex colors. You may use ANY color palette (pastel, earthy, muted, bright, warm, cool, mixed, etc.) as long as the colors are visually easy to tell apart. Choose one diff_index (0 to 3). Provide a diff_color that is clearly different from the ORIGINAL color at that exact index. Do not compare diff_color against the other 3 slots. Output Format (Raw JSON only): {"grid_colors": ["#hex1", "#hex2", "#hex3", "#hex4"], "diff_index": 0, "diff_color": "#hex_new" }'''
logic_beginner_prompt_2 = '''You are an Activity Generator for 4-year-olds. Create a 'Spot the Difference' game. TASK: Pick a fresh, creative theme each time. Generate a 2x2 color grid. RULES: - Provide grid_colors as 4 clearly distinct Hex colors that preschoolers can tell apart quickly. - Any palette is allowed (including red/green/blue/yellow combinations) if the colors are clear. - Avoid forcing a standardized neon look; vary styles naturally across generations. DIFFERENCE RULE: - Choose one diff_index (0-3). - diff_color must be judged against the replaced color at that diff_index only, and should be clearly different from that one. - Keep JSON simple and clean. OUTPUT ONLY RAW JSON: { "grid_colors": ["#hex1", "#hex2", "#hex3", "#hex4"], "diff_index": 0, "diff_color": "#hex_new" }'''

logic_intermediate_prompt = '''You are an Activity Generator. Create a 'Spot the Difference' 3x3 grid game for a 6-year-old. Choose a playful theme (must influence ALL colors deeply, e.g. Jungle, Candy, Aquarium, Winter, Bubblegum, Toy Ã¢â‚¬â€ be creative and NEVER reuse similar themes repeatedly). Provide a list of 9 Hex colors for the first grid. Can be pastel, muted colouors, etc. must all be different colours and not shades of the same colour family. RULES: - All 9 colors MUST belong to the same visual theme family. - Each color must represent a different object or element inside the theme (not random colors). - Do NOT use generic rainbow sets or unrelated primary color lists. - Colors must feel like they belong in the same world/environment. DIFFERENCE RULE: - Choose one index (0Ã¢â‚¬â€œ8). - Replace it with a diff_color that comes from a COMPLETELY DIFFERENT object in the same theme. - It MUST NOT be a shade variation. OUTPUT ONLY RAW JSON: { "grid_colors": ["#hex1", "#hex2", ...], "diff_index": 4, "diff_color": "#hex_new" }'''

logic_hard_prompt = '''Advanced visual discrimination of luminance and saturation. System Role: You are an Educational Content Creator for preschool computational thinking. Your goal is to generate a "Spot the Difference" logic puzzle for 6-year-olds. Task: Generate a 3x3 grid of colors based on a sophisticated monochromatic theme. Rules for "Hard" Difficulty: Monochromatic Theme: All 10 colors (9 in grid + 1 difference) must belong to the same color family (e.g., all Ocean Blues or all Forest Greens or baby pastel, or browns. each generation should not repeat the same colour.). No unrelated colors allowed. The "Fairness" Rule: the diff_color should still fit the same color family, but it only needs to be clearly distinguishable to a preschool eye. It should not be an exact or near-exact match, but it also does not need to jump to a totally different family. If the grid is "Pastel," the difference can be deeper or darker. If the grid is dark, the difference can be lighter. Keep the change noticeable without making the puzzle unfair. Make sure all the colours, even the diff_color is under the same colour family. dont be like a dark colour becomes white. Randomize Placement: Jumble the hex codes so there is no predictable gradient (e.g., no lightest-to-darkest sorting). Negative Constraint: Do not include conversational text or Markdown formatting (no ```json). Output raw JSON only. Output JSON Structure: { "grid_colors": ["#hex1", "#hex2", "#hex3", "#hex4", "#hex5", "#hex6", "#hex7", "#hex8", "#hex9"], "diff_index": 0, "diff_color": "#hex_different_shade" }. all colours in the 3x3 can be any random shade INSIDE the colour family. a neon and a pastel can be shown. make sure that all colours are not too similar so its very distinct.'''

logic_beginner_prompts = [logic_beginner_prompt_1, logic_beginner_prompt_2, logic_beginner_prompt_1]
logic_intermediate_prompts = [logic_intermediate_prompt, logic_intermediate_prompt, logic_intermediate_prompt]
logic_hard_prompts = [logic_hard_prompt, logic_hard_prompt, logic_hard_prompt]

In [353]:
def _generate_logic_payload_relaxed(prompt, filename, temperature):
    return generate_openai_payload(prompt, filename, temperature=temperature)


test_data_beginner_1 = _generate_logic_payload_relaxed(logic_beginner_prompts[0], "logic_beginner_1.json", temperature=0.95)
test_data_beginner_2 = _generate_logic_payload_relaxed(logic_beginner_prompts[1], "logic_beginner_2.json", temperature=0.95)
test_data_beginner_3 = _generate_logic_payload_relaxed(logic_beginner_prompts[2], "logic_beginner_3.json", temperature=0.95)
test_data_intermediate_1 = _generate_logic_payload_relaxed(logic_intermediate_prompts[0], "logic_intermediate_1.json", temperature=0.9)
test_data_intermediate_2 = _generate_logic_payload_relaxed(logic_intermediate_prompts[1], "logic_intermediate_2.json", temperature=0.9)
test_data_intermediate_3 = _generate_logic_payload_relaxed(logic_intermediate_prompts[2], "logic_intermediate_3.json", temperature=0.9)
test_data_hard_1 = _generate_logic_payload_relaxed(logic_hard_prompts[0], "logic_hard_1.json", temperature=0.8)
test_data_hard_2 = _generate_logic_payload_relaxed(logic_hard_prompts[1], "logic_hard_2.json", temperature=0.8)
test_data_hard_3 = _generate_logic_payload_relaxed(logic_hard_prompts[2], "logic_hard_3.json", temperature=0.8)


def load_font(size, bold=False):
    font_paths = [
        r"C:\Windows\Fonts\arialbd.ttf" if bold else r"C:\Windows\Fonts\arial.ttf",
        r"C:\Windows\Fonts\segoeuib.ttf" if bold else r"C:\Windows\Fonts\segoeui.ttf",
    ]
    for font_path in font_paths:
        if os.path.exists(font_path):
            try:
                return ImageFont.truetype(font_path, size=size)
            except Exception:
                pass
    return ImageFont.load_default()


def _generate_logic_worksheet(activity_func, test_jsons, activity_name, level_label, instruction_text):
    return create_printable_worksheet(
        activity_func,
        test_jsons,
        activity_name=activity_name,
        level_label=level_label,
        instruction_text=instruction_text,
        question_prompt="Circle the different answer."
    )


def generate_beginner_worksheet():
    return _generate_logic_worksheet(
        generate_beginner_spot_diff,
        [test_data_beginner_1, test_data_beginner_2, test_data_beginner_3],
        "BEGINNER_LOGIC_PROBLEM_SOLVING",
        "Beginner",
        "Instructions: Find the odd one out in each set. Circle the answer.",
    )


def generate_intermediate_worksheet():
    return _generate_logic_worksheet(
        generate_spot_the_diff_intermediate,
        [test_data_intermediate_1, test_data_intermediate_2, test_data_intermediate_3],
        "INTERMEDIATE_LOGIC_PROBLEM_SOLVING",
        "Intermediate",
        "Instructions: Compare each pair carefully and circle the one that is different.",
    )


def generate_hard_worksheet():
    return _generate_logic_worksheet(
        generate_spot_the_diff_hard,
        [test_data_hard_1, test_data_hard_2, test_data_hard_3],
        "HARD_LOGIC_PROBLEM_SOLVING",
        "Hard",
        "Instructions: Study each set closely. Circle the answer that does not match the others.",
    )

#generate
generate_beginner_worksheet()
generate_intermediate_worksheet()
generate_hard_worksheet()

No activities generated!
No activities generated!
No activities generated!


## Sequencing / Size Ordering
Sequencing prompts, generators, and worksheet rendering.

In [354]:
# Size Ordering activity generator (labels removed, emojis auto-scaled to fit)
from PIL import Image, ImageDraw
import os


def _load_emoji_font(size):
    candidates = [
        r"C:\Windows\Fonts\seguiemj.ttf",
        r"C:\Windows\Fonts\SegoeUIEmoji.ttf",
        r"C:\Windows\Fonts\AppleColorEmoji.ttf",
        r"/System/Library/Fonts/AppleColorEmoji.ttf",
        "NotoColorEmoji.ttf",
    ]
    for font_path in candidates:
        try:
            return ImageFont.truetype(font_path, size=size)
        except Exception:
            continue
    return ImageFont.load_default()


def generate_size_ordering_from_llm(llm_json):
    """Generates a size-ordering sequencing activity from LLM JSON.
    Removes label text and scales emoji to fit inside the card.
    Expected format:
    {"theme":"...","jumbled_sequence":[{"emoji":"Ã°Å¸Â§Â¸","size_rating":1}, ...], "correct_index_order":[...]} 
    """
    jseq = llm_json.get('jumbled_sequence', [])
    if not jseq:
        print('No jumbled_sequence provided')
        return None

    card_size = llm_json.get('card_size', 320)
    spacing = llm_json.get('spacing', 30)
    answer_box_w = llm_json.get('answer_box_w', 62)
    answer_box_h = llm_json.get('answer_box_h', 46)

    total_width = (card_size * len(jseq)) + (spacing * (len(jseq) + 1))
    total_height = card_size + (spacing * 2) + answer_box_h + 12

    transparent = llm_json.get('transparent', False)
    if transparent:
        canvas = Image.new('RGBA', (total_width, total_height), (255,255,255,0))
    else:
        canvas = Image.new('RGB', (total_width, total_height), 'white')
    draw = ImageDraw.Draw(canvas)

    # size_rating 1..5 -> initial emoji sizes
    size_map = {
        1: int(card_size * 0.18),
        2: int(card_size * 0.30),
        3: int(card_size * 0.44),
        4: int(card_size * 0.60),
        5: int(card_size * 0.92),
    }

    x = spacing
    for idx, item in enumerate(jseq):
        glyph = item.get('emoji', '')
        rating = int(item.get('size_rating', 3))
        target_px = size_map.get(rating, size_map[3])

        # draw card
        left, top = x, spacing
        right, bottom = x + card_size, spacing + card_size
        draw.rectangle([left, top, right, bottom], fill='white' if not transparent else (255,255,255,0), outline='black', width=3)

        # center
        cx = left + card_size / 2
        cy = top + card_size / 2

        # compute max drawable area inside card (leave padding)
        pad = max(6, int(card_size * 0.06))
        max_w = card_size - pad * 2
        max_h = card_size - pad * 2

        # Find max fitting size for this glyph, then tier down from that.
        max_fit = max(10, int(card_size * 0.92))
        probe = max_fit
        while probe > 8:
            try:
                probe_font = _load_emoji_font(probe)
                probe_bbox = draw.textbbox((0,0), glyph, font=probe_font)
                probe_w = probe_bbox[2] - probe_bbox[0]
                probe_h = probe_bbox[3] - probe_bbox[1]
                if probe_w <= max_w and probe_h <= max_h:
                    max_fit = probe
                    break
                probe = int(probe * 0.92)
            except Exception:
                probe = probe - 2

        tier_scale = {1: 0.34, 2: 0.50, 3: 0.66, 4: 0.80, 5: 1.00}
        target_px = max(10, int(max_fit * tier_scale.get(rating, 0.66)))

        # attempt to find an emoji font size that fits within max_w x max_h
        ef_px = target_px
        ef = None
        while ef_px > 8:
            try:
                ef_candidate = _load_emoji_font(ef_px)
                bbox = draw.textbbox((0,0), glyph, font=ef_candidate)
                w = bbox[2] - bbox[0]
                h = bbox[3] - bbox[1]
                if w <= max_w and h <= max_h:
                    ef = ef_candidate
                    break
                # reduce size and retry
                ef_px = int(ef_px * 0.9)
            except Exception:
                ef_px = ef_px - 2
        if ef is None:
            # fallback to a small default
            ef = _load_emoji_font(int(card_size * 0.30))

        # draw emoji centered
        try:
            draw.text((cx, cy), glyph, font=ef, anchor='mm', fill='black')
        except Exception:
            # last-resort draw
            draw.text((cx, cy), glyph, anchor='mm', fill='black')

        # small centered answer box below card
        bx = int(cx - answer_box_w/2)
        by = int(bottom + 8)
        draw.rectangle([bx, by, bx + answer_box_w, by + answer_box_h], outline='black', width=3, fill='white' if not transparent else (255,255,255,0))

        x += card_size + spacing

    # Save
    os.makedirs('activities/sequencing', exist_ok=True)
    file_number = 1
    output_path = f"activities/sequencing/size_ordering_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/sequencing/size_ordering_{file_number}.png"
    canvas.save(output_path)
    print(f"Ã¢Å“â€œ Saved: {output_path}")
    return output_path


In [ ]:
_original_generate_size_ordering_from_llm = generate_size_ordering_from_llm


def _sequencing_payload_is_skipped(llm_json):
    return not llm_json or (isinstance(llm_json, dict) and llm_json.get("__skip__"))


def generate_size_ordering_from_llm(llm_json):
    if _sequencing_payload_is_skipped(llm_json):
        return None
    return _original_generate_size_ordering_from_llm(llm_json)


## Sequencing Worksheet Builder
Worksheet image generation and printable page layout.

In [356]:
# WORKSHEET GENERATOR FOR SEQUENCING - 3 of same activity per sheet
from PIL import Image, ImageDraw, ImageFont
import os
import random

sequencing_beginner_base = """BEGINNER:
Create a simple size-sequencing JSON puzzle for a preschooler.
Use exactly 3 items.
Use the same emoji in every item. Different emojis are not allowed.
Each jumbled_sequence item must use the keys emoji and size_rating only.
Use distinct size_rating values (integers 1 to 5).
Jumble the shown order so it is not already smallest-to-biggest.
Set correct_index_order as per-item ranks in shown order: 1=smallest, N=largest.
Return raw JSON only with theme, jumbled_sequence, and correct_index_order.
"""
sequencing_intermediate_base = """INTERMEDIATE:
Create a simple size-sequencing JSON puzzle for a preschooler.
Use exactly 4 items.
Use the same emoji in every item. Different emojis are not allowed.
Each jumbled_sequence item must use the keys emoji and size_rating only.
Use distinct size_rating values (integers 1 to 5).
Jumble the shown order so it is not already smallest-to-biggest.
Set correct_index_order as per-item ranks in shown order: 1=smallest, N=largest.
Return raw JSON only with theme, jumbled_sequence, and correct_index_order.
"""
sequencing_hard_base = """HARD:
Create a more challenging size-sequencing JSON puzzle for a preschooler.
Use exactly 5 items.
Use the same emoji in every item. Different emojis are not allowed.
Each jumbled_sequence item must use the keys emoji and size_rating only.
Use distinct size_rating values (integers 1 to 5).
Jumble the shown order so it is not already smallest-to-biggest.
Set correct_index_order as per-item ranks in shown order: 1=smallest, N=largest.
Return raw JSON only with theme, jumbled_sequence, and correct_index_order.
"""

sequencing_beginner_prompt_1 = sequencing_beginner_base + " Theme: animals."
sequencing_beginner_prompt_2 = sequencing_beginner_base + " Theme: toys."
sequencing_beginner_prompt_3 = sequencing_beginner_base + " Theme: food."
sequencing_intermediate_prompt_1 = sequencing_intermediate_base + " Theme: animals."
sequencing_intermediate_prompt_2 = sequencing_intermediate_base + " Theme: shapes."
sequencing_intermediate_prompt_3 = sequencing_intermediate_base + " Theme: nature."
sequencing_hard_prompt_1 = sequencing_hard_base + " Theme: animals."
sequencing_hard_prompt_2 = sequencing_hard_base + " Theme: shapes."
sequencing_hard_prompt_3 = sequencing_hard_base + " Theme: nature."

SEQUENCING_EMOJI_POOLS = {
    "beginner": ["Ã°Å¸ÂÂ»", "Ã°Å¸ÂÂ¸", "Ã°Å¸Â¦Å ", "Ã°Å¸ÂÂ¶", "Ã°Å¸ÂÂ¼", "Ã°Å¸ÂÂµ", "Ã°Å¸Â¦Â", "Ã°Å¸ÂÂ°", "Ã°Å¸ÂÂ§", "Ã°Å¸Â¦â€ž"],
    "intermediate": ["Ã¢Â­Â", "Ã°Å¸Å’â„¢", "Ã¢Ëœâ‚¬Ã¯Â¸Â", "Ã°Å¸Å’Ë†", "Ã°Å¸ÂªÂ", "Ã°Å¸Å¡â€š", "Ã°Å¸Â§Â©", "Ã°Å¸Å¡â€”", "Ã°Å¸Å½Ë†", "Ã°Å¸Â§Â¸"],
    "hard": ["Ã°Å¸Å’Â³", "Ã°Å¸ÂÅ½", "Ã°Å¸Ââ€¡", "Ã°Å¸Â¥â€¢", "Ã°Å¸Å’Â¸", "Ã°Å¸Ââ‚¬", "Ã°Å¸ÂªÂµ", "Ã°Å¸Â¦â€¹", "Ã°Å¸Ââ„¢", "Ã°Å¸ÂÂ¢"],
}

SEQUENCING_ORDER_TEMPLATES = {
    "beginner": [[1, 2, 0], [2, 0, 1], [1, 0, 2]],
    "intermediate": [[1, 3, 0, 2], [2, 0, 3, 1], [3, 1, 0, 2]],
    "hard": [[1, 4, 0, 3, 2], [3, 0, 4, 1, 2], [2, 4, 1, 0, 3]],
}

def _enforce_sequencing_uniqueness(payload, level_name, sample_index, emoji_choices):
    if not isinstance(payload, dict):
        return payload
    jseq = payload.get('jumbled_sequence')
    if not isinstance(jseq, list) or not jseq:
        return payload

    items = [dict(item) if isinstance(item, dict) else {} for item in jseq]
    ratings = [item.get('size_rating') for item in items]
    if any(not isinstance(rating, int) for rating in ratings):
        return payload

    sorted_items = [dict(items[idx]) for idx in sorted(range(len(items)), key=lambda idx: ratings[idx])]
    templates = SEQUENCING_ORDER_TEMPLATES.get(level_name, [])
    if templates:
        template = templates[sample_index % len(templates)]
        if len(template) == len(sorted_items):
            reordered = [dict(sorted_items[idx]) for idx in template]
        else:
            reordered = list(reversed(sorted_items))
    else:
        reordered = list(reversed(sorted_items))

    label_lookup = {1: 'Tiny', 2: 'Small', 3: 'Medium', 4: 'Large', 5: 'Huge'}
    emoji_value = emoji_choices[sample_index % len(emoji_choices)] if emoji_choices else 'Ã¢Â­Â'
    for item in reordered:
        item['emoji'] = emoji_value
        rating = item.get('size_rating')
        if isinstance(rating, int) and rating in label_lookup:
            item['label'] = label_lookup[rating]

    size_ratings = [item.get('size_rating') for item in reordered]
    sorted_ratings = sorted(size_ratings)
    rank_lookup = {rating: rank for rank, rating in enumerate(sorted_ratings, start=1)}
    computed_order = [rank_lookup[rating] for rating in size_ratings]
    updated = dict(payload)
    updated['jumbled_sequence'] = reordered
    updated['correct_index_order'] = computed_order
    return updated

def _generate_unique_sequencing_set(level_name, prompts, filenames, temperature):
    emoji_choices = list(SEQUENCING_EMOJI_POOLS.get(level_name, ['Ã¢Â­Â']))
    random.shuffle(emoji_choices)
    outputs = []
    for sample_index, (prompt_text, filename) in enumerate(zip(prompts, filenames)):
        payload = generate_openai_payload(prompt_text, filename, temperature=temperature)
        payload = _enforce_sequencing_uniqueness(payload, level_name, sample_index, emoji_choices)
        save_json_output('activities/json_outputs', filename, payload)
        outputs.append(payload)
    return outputs

sequencing_beginner_1, sequencing_beginner_2, sequencing_beginner_3 = _generate_unique_sequencing_set(
    'beginner',
    [sequencing_beginner_prompt_1, sequencing_beginner_prompt_2, sequencing_beginner_prompt_3],
    ['sequencing_beginner_1.json', 'sequencing_beginner_2.json', 'sequencing_beginner_3.json'],
    0.4,
)
sequencing_intermediate_1, sequencing_intermediate_2, sequencing_intermediate_3 = _generate_unique_sequencing_set(
    'intermediate',
    [sequencing_intermediate_prompt_1, sequencing_intermediate_prompt_2, sequencing_intermediate_prompt_3],
    ['sequencing_intermediate_1.json', 'sequencing_intermediate_2.json', 'sequencing_intermediate_3.json'],
    0.4,
)
sequencing_hard_1, sequencing_hard_2, sequencing_hard_3 = _generate_unique_sequencing_set(
    'hard',
    [sequencing_hard_prompt_1, sequencing_hard_prompt_2, sequencing_hard_prompt_3],
    ['sequencing_hard_1.json', 'sequencing_hard_2.json', 'sequencing_hard_3.json'],
    0.35,
)


def _load_text_font(size):
    candidates = [
        r"C:\Windows\Fonts\arial.ttf",
        r"C:\Windows\Fonts\segoeui.ttf",
    ]
    for p in candidates:
        try:
            return ImageFont.truetype(p, size=size)
        except:
            pass
    return ImageFont.load_default()


def _load_emoji_font(size):
    candidates = [
        r"C:\Windows\Fonts\seguiemj.ttf",
        r"C:\Windows\Fonts\SegoeUIEmoji.ttf",
        r"C:\Windows\Fonts\AppleColorEmoji.ttf",
        r"/System/Library/Fonts/AppleColorEmoji.ttf",
        "NotoColorEmoji.ttf",
    ]
    for p in candidates:
        try:
            return ImageFont.truetype(p, size=size)
        except:
            continue
    return ImageFont.load_default()


def generate_sequencing_from_llm(llm_json):
    jseq = llm_json.get('jumbled_sequence', [])
    if not jseq:
        print('No jumbled_sequence provided')
        return None

    card_size = llm_json.get('card_size', 200)
    spacing = llm_json.get('spacing', 20)
    answer_box_w = llm_json.get('answer_box_w', 48)
    answer_box_h = llm_json.get('answer_box_h', 36)

    total_width = (card_size * len(jseq)) + (spacing * (len(jseq) + 1))
    total_height = card_size + (spacing * 2) + answer_box_h + 12

    transparent = llm_json.get('transparent', False)
    if transparent:
        canvas = Image.new('RGBA', (total_width, total_height), (255, 255, 255, 0))
    else:
        canvas = Image.new('RGB', (total_width, total_height), 'white')
    draw = ImageDraw.Draw(canvas)

    label_font = _load_text_font(int(card_size * 0.10))
    emoji_font = _load_emoji_font(int(card_size * 0.60))

    x = spacing
    for item in jseq:
        glyph = item.get('emoji', '')
        label = item.get('label', '')

        left, top = x, spacing
        right, bottom = x + card_size, spacing + card_size
        draw.rectangle([left, top, right, bottom], fill='white' if not transparent else (255, 255, 255, 0), outline='black', width=3)

        cx = left + card_size / 2
        cy = top + card_size / 2

        if glyph:
            try:
                bbox = draw.textbbox((cx, cy), glyph, font=emoji_font, anchor='mm')
            except Exception:
                bbox = None
            if bbox:
                gx0, gy0, gx1, gy1 = bbox
                pad = max(6, int(card_size * 0.04))
                min_x = left + pad
                max_x = right - pad
                shift_x = 0
                if gx0 < min_x:
                    shift_x = min_x - gx0
                elif gx1 > max_x:
                    shift_x = max_x - gx1
                min_y = top + pad
                max_y = bottom - pad
                shift_y = 0
                if gy0 < min_y:
                    shift_y = min_y - gy0
                elif gy1 > max_y:
                    shift_y = max_y - gy1
                draw.text((cx + shift_x, cy + shift_y), glyph, font=emoji_font, anchor='mm', fill='black')
            else:
                draw.text((cx, cy), glyph, font=emoji_font, anchor='mm', fill='black')

        if label:
            label_y = top + int(card_size * 0.82)
            draw.text((cx, label_y), label, font=label_font, anchor='mm', fill='black')

        bx = int(cx - answer_box_w / 2)
        by = int(bottom + 8)
        draw.rectangle([bx, by, bx + answer_box_w, by + answer_box_h], outline='black', width=3, fill='white' if not transparent else (255, 255, 255, 0))

        x += card_size + spacing

    os.makedirs('activities/sequencing', exist_ok=True)
    file_number = 1
    output_path = f"activities/sequencing/sequencing_activity_{file_number}.png"
    while os.path.exists(output_path):
        file_number += 1
        output_path = f"activities/sequencing/sequencing_activity_{file_number}.png"

    canvas.save(output_path)
    print(f"Ã¢Å“â€œ Saved: {output_path}")
    return output_path


# size-ordering worksheet (3 copies) for each level
def create_printable_worksheet(activity_func, test_jsons, activity_name="worksheet", level_label="LEVEL", instruction_text="", question_prompt="Circle the different answer.", out_dir='activities/worksheets', spacing=18):
    os.makedirs(out_dir, exist_ok=True)

    activity_paths = []
    for test_json in test_jsons:
        path = None
        if isinstance(test_json, dict):
            cached_path = test_json.get("__image_path")
            if isinstance(cached_path, str) and cached_path and os.path.exists(cached_path):
                path = cached_path

        if path is None:
            path = activity_func(test_json)
            if path and isinstance(test_json, dict):
                test_json["__image_path"] = path

        if path:
            activity_paths.append(path)

    if not activity_paths:
        print("No activities generated!")
        return None

    max_activity_width = 1300
    imgs = []
    for path in activity_paths:
        im = Image.open(path).convert('RGBA')
        if im.width > max_activity_width:
            new_height = int(im.height * (max_activity_width / im.width))
            im = im.resize((max_activity_width, new_height), Image.Resampling.LANCZOS)
        imgs.append(im)

    page_width = 1900
    margin_x = 100
    header_height = 260
    prompt_height = 60
    question_pad = 20

    page_height = header_height + sum(im.height + prompt_height + (question_pad * 2) for im in imgs) + spacing * (len(imgs) - 1) + 60
    canvas = Image.new('RGBA', (page_width, page_height), (255, 255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    title_font = _load_text_font(50)
    label_font = _load_text_font(32)
    body_font = _load_text_font(30)
    prompt_font = _load_text_font(32)

    draw.rectangle([0, 0, page_width, header_height], fill=(255, 255, 255, 255))

    title = activity_name.replace('_', ' ').upper()
    draw.text((margin_x, 24), title, fill=(25, 25, 25), font=title_font, anchor='la')
    draw.text((margin_x, 92), f"Level: {level_label}", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.text((margin_x + 380, 92), "Name: ____________________________", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.text((margin_x + 1040, 92), "Date: __________________", fill=(45, 45, 45), font=label_font, anchor='la')
    draw.multiline_text((margin_x, 148), instruction_text or "Instructions: Look carefully at each set and circle the one that is different.", fill=(45, 45, 45), font=body_font, spacing=5)
    draw.text((margin_x, 202), "Complete all 3 questions on this page.", fill=(45, 45, 45), font=body_font, anchor='la')

    y = header_height + 22
    for index, im in enumerate(imgs, start=1):
        block_top = y
        block_bottom = y + prompt_height + im.height + (question_pad * 2)

        draw.rounded_rectangle([margin_x - 14, block_top, page_width - margin_x + 14, block_bottom], radius=22, outline=(210, 210, 210), width=2, fill=(255, 255, 255, 255))

        prompt = f"Question {index}. {question_prompt}"
        draw.text((margin_x + 16, block_top + 16), prompt, fill=(30, 30, 30), font=prompt_font, anchor='la')

        image_y = block_top + prompt_height + question_pad
        image_x = (page_width - im.width) // 2
        canvas.paste(im, (image_x, image_y), im)

        y = block_bottom + spacing

    out_path = os.path.join(out_dir, f'{activity_name}.png')
    canvas.save(out_path)
    print(f"Worksheet saved: {out_path}")
    print(f"Worksheet size: {canvas.size[0]}x{canvas.size[1]}")
    return out_path


def _generate_sequencing_worksheet(activity_name, level_label, sequence_items):
    test_jsons = [
        {**sequence_items[0], "card_size": 320, "spacing": 30, "answer_box_w": 62, "answer_box_h": 46},
        {**sequence_items[1], "card_size": 320, "spacing": 30, "answer_box_w": 62, "answer_box_h": 46},
        {**sequence_items[2], "card_size": 320, "spacing": 30, "answer_box_w": 62, "answer_box_h": 46},
    ]
    return create_printable_worksheet(
        generate_size_ordering_from_llm,
        test_jsons,
        activity_name=activity_name,
        level_label=level_label,
        instruction_text="Instructions: Write down the number of sizing in the rectangle blank below. From smallest to biggest (1 = smallest).",
        question_prompt="Write down the number of sizing in the rectangle blank below."
    )


def generate_beginner_worksheet():
    return _generate_sequencing_worksheet(
        "BEGINNER_Sequencing",
        "Beginner",
        [sequencing_beginner_1, sequencing_beginner_2, sequencing_beginner_3],
    )


def generate_intermediate_worksheet():
    return _generate_sequencing_worksheet(
        "INTERMEDIATE_Sequencing",
        "Intermediate",
        [sequencing_intermediate_1, sequencing_intermediate_2, sequencing_intermediate_3],
    )


def generate_hard_worksheet():
    return _generate_sequencing_worksheet(
        "HARD_Sequencing",
        "Hard",
        [sequencing_hard_1, sequencing_hard_2, sequencing_hard_3],
    )


generate_beginner_worksheet()
generate_intermediate_worksheet()
generate_hard_worksheet()

No activities generated!
No activities generated!
No activities generated!
